# Homework 04 - Evaluation

In [1]:
import json

from pydantic import BaseModel, Field
from typing import Literal

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

from embedder import Embedder
from evaluation_utils import llm_structured


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
len(documents)

72

In [4]:
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
class Questions(BaseModel):
    questions: list[str]

In [7]:
openai_client = OpenAI()
records = []
input_tokens = 0

for doc in documents[:3]:
    user_prompt = json.dumps(doc)

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    print(result.questions)
    for q in result.questions:
        records.append({
            "filename": doc["filename"],
            "question": q
        })
    input_tokens += usage.input_tokens


['What exactly is RAG, and how does it help with LLM limitations like stale knowledge or missing access to my files?', 'Why does the lesson say to treat the LLM like a black box instead of trying to run or inspect one ourselves?', 'What kind of app are we building in this module to demonstrate RAG?', 'What will be covered in the first part of the module before the agentic version starts?', 'What changes in the second part when the LLM becomes agentic?']
['What do I need installed before I can follow this module, and do I need anything besides Python and Jupyter?', 'How do I set up a fresh project for this course from zero, including the package manager and the needed libraries?', 'What’s the safest way to keep my API key out of git when using this setup?', 'How do I open Jupyter and check that the OpenAI client is working in a notebook?', 'If I want to use Groq or another OpenAI-style API instead of OpenAI, how should I configure the client?']
['Why doesn’t a normal LLM give a good ans

In [8]:
# Calculate the average number of tokens 

average_input_tokens = input_tokens/3
average_input_tokens

1353.0

# Question 1 - What's the average number of input tokens across these 3 calls?

Average number of input tokens: 1353

## Answer: 1400 

In [9]:
len(records)

15

In [10]:
#PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"

In [11]:
# Load ground truth records

df_ground_truth = pd.read_csv("ground-truth.csv")

df_ground_truth.head()


,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [12]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [13]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

In [16]:
# Create text search index
from minsearch import Index

text_search_index = Index(
    text_fields=['content', 'filename'],
    keyword_fields=['filename']
)
text_search_index.fit(chunks)


In [27]:
def text_search(query:str, num_results:int = 5):
    boost = {'content': 3.0}

    return text_search_index.search(
        query,
        num_results=num_results,
        boost_dict=boost
    )

In [31]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [32]:
text_results = text_search(q)
text_results

[{'start': 4000,
  'content': 'ogress\n```\n\nRun RAG for all ground truth questions:\n\n```python\nwith ThreadPoolExecutor(max_workers=6) as pool:\n    results = map_progress(pool, ground_truth, generate_rag_answer)\n```\n\n`generate_rag_answer` returns one answer record for each question.\n\nCollect the answer records:\n\n```python\nanswers = []\n\nfor answer_record in results:\n    answers.append(answer_record)\n```\n\nCalculate the total cost:\n\n```python\nassistant.total_cost()\n```\n\nSave the answers:\n\n```python\ndf_answers = pd.DataFrame(answers)\ndf_answers.to_csv("data/rag-answers-new.csv", index=False)\n```\n\nWe generated this file for the course materials on May 29, 2026. The\nrun used 395 ground truth questions.\n\nThe total cost was $0.34332825, about 34 cents.\n\nIf you don\'t want to generate the RAG answers yourself, download the\nfile we prepared:\n\n```bash\nPREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main\n\nwget -O data/rag-answers-new.c

# Question 2: First result with text search
## Answer: 01-agentic-rag/lessons/03-rag.md

In [19]:
# Create embedded store of chunked documents 

embed = Embedder()

texts = [doc["filename"] + " " + doc["content"] for doc in chunks]
batch_size = 50
chunks_embedded = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    chunks_embedded.extend(batch_vectors)

chunks_embedded = np.array(chunks_embedded)

  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
from minsearch import VectorSearch

vector_search_index = VectorSearch(keyword_fields=["filename"])
vector_search_index.fit(chunks_embedded, chunks)


In [21]:
def vector_search(query:str, num_results:int = 5):
    query_vector = embed.encode(query)
    return vector_search_index.search(query_vector, num_results=num_results)

In [30]:
vector_results = vector_search(q)
vector_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

# Question 3: First result with vector search
## Answer: 01-agentic-rag/lessons/01-intro.md

In [33]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [34]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
# Calculate the hit rate for text search using ground truth
cnt = 0
for q in ground_truth:
    results = text_search(q['question'])
    if q['filename'] in [result['filename'] for result in results]:
        cnt += 1


hit_rate = cnt / len(ground_truth)
hit_rate


0.6305555555555555

# Question 4: Evaluating text search
## Answer: 0.66

In [38]:
# Calculate MRR for vector search
total_score = 0

for q in ground_truth:
    results = vector_search(q['question'])
    for rank in range(len(results)):
        if results[rank]['filename'] == q['filename']:
            total_score = total_score + 1 / (rank + 1)
            break

mrr = total_score / len(ground_truth)
mrr


0.5469907407407407

# Question 5: Evaluating vector search
## Answer: 0.55

In [39]:


for q in ground_truth:
    rff_result = hybrid_search(q['question'], k=60)
    break
rff_result

[{'start': 3000,
  'content': '# Processing all questions\n\nCreate a function that processes one ground truth record:\n\n```python\ndef generate_rag_answer(rec):\n    question = rec["question"]\n    doc_id = rec["document"]\n    original_doc = doc_idx[doc_id]\n\n    answer_llm = assistant.rag(question)\n    answer_orig = original_doc["answer"]\n\n    result = {\n        "question": question,\n        "answer_llm": answer_llm,\n        "answer_orig": answer_orig,\n        "document": doc_id,\n    }\n\n    return result\n```\n\nTest it on one record:\n\n```python\nanswer_record = generate_rag_answer(ground_truth[0])\nanswer_record\n```\n\nBefore running the full batch, reset the usage we collected while\ntesting:\n\n```python\nassistant.reset_usage()\n```\n\nThis calls the LLM once per ground truth question, so it can take some\ntime. Let\'s process the questions in parallel and track progress.\n\nImport the parallel processing helper from the same utility file:\n\n```python\nfrom concu

In [41]:
# Calculate MRR for hybrid search 

for K in [1, 50, 100, 200]:
    total_score = 0

    for q in ground_truth:
        results = hybrid_search(q['question'], k=K)
        for rank in range(len(results)):
            if results[rank]['filename'] == q['filename']:
                total_score = total_score + 1 / (rank + 1)
                break

    mrr = total_score / len(ground_truth)
    print(f"k: {K} MRR: {mrr}")

k: 1 MRR: 0.5872685185185186
k: 50 MRR: 0.5759259259259261
k: 100 MRR: 0.5759259259259261
k: 200 MRR: 0.5759259259259261


# Question 6: Tuning hybrid search
## Answer: 1 